In [1]:
import numpy as np
import sys
 
def generate_pure_polymer(beads_per_chain=5, units_x=8, units_y=8, units_z=8,
                          output_file="pure_polymer.data"):
    """
    Generate a periodic tetrahedral (diamond lattice) crosslinked polymer gel
    with no solvent, no support, and no piston.
 
    Atom types:
      1 - Crosslink bead
      2 - Chain bead
 
    Bond types:
      1 - FENE chain bond
 
    The simulation box is exactly the gel dimensions so the network tiles
    periodically in all three directions.
 
    Parameters:
    - beads_per_chain: number of beads per chain (including the two crosslink endpoints)
    - units_x, units_y, units_z: number of diamond unit cells in x, y, z
    - output_file: LAMMPS data file name
    """
 
    # -----------------------------------------------------------------------
    # Lattice geometry
    # -----------------------------------------------------------------------
    bead_spacing = 1.2                          # initial bond length (σ)
    chain_length = bead_spacing * (beads_per_chain - 1)
    a = chain_length                            # diamond FCC lattice constant
 
    # Box = gel dimensions (fully periodic, no padding needed)
    box_x = units_x * a
    box_y = units_y * a
    box_z = units_z * a
 
    # No offset: gel fills the entire box
    offset = np.array([0.0, 0.0, 0.0])
 
    particles = []
    bonds     = []
    particle_id = 1
    bond_id     = 1
    molecule_id = 1
 
    # -----------------------------------------------------------------------
    # Crosslink positions (diamond lattice)
    # Diamond = 2 interpenetrating FCC sub-lattices offset by (a/4, a/4, a/4)
    # -----------------------------------------------------------------------
    crosslinks = {}
 
    fcc_basis = [
        np.array([0.0, 0.0, 0.0]),
        np.array([0.5, 0.5, 0.0]),
        np.array([0.5, 0.0, 0.5]),
        np.array([0.0, 0.5, 0.5]),
    ]
 
    for i in range(units_x + 1):
        for j in range(units_y + 1):
            for k in range(units_z + 1):
                for sublattice in [0, 1]:
                    for fcc_idx, fcc_frac in enumerate(fcc_basis):
                        if sublattice == 1:
                            frac_pos = fcc_frac + np.array([0.25, 0.25, 0.25])
                        else:
                            frac_pos = fcc_frac
 
                        pos = (np.array([i, j, k]) + frac_pos) * a + offset
 
                        # Wrap into [0, box] so duplicates across PBC are removed
                        wrapped = pos % np.array([box_x, box_y, box_z])
 
                        # Use rounded key to deduplicate wrapped positions
                        key = tuple(np.round(wrapped, 6))
 
                        if key in crosslinks:
                            continue
                        # Keep only sites inside [0, box)
                        if (0 <= wrapped[0] < box_x and
                                0 <= wrapped[1] < box_y and
                                0 <= wrapped[2] < box_z):
                            crosslinks[key] = {
                                'id': particle_id,
                                'pos': wrapped.copy(),
                            }
                            particles.append({
                                'id':   particle_id,
                                'type': 1,        # Crosslink
                                'pos':  wrapped.copy(),
                                'mol':  0,
                            })
                            particle_id += 1
 
    # -----------------------------------------------------------------------
    # Chain beads + bonds (nearest-neighbour crosslink pairs)
    # Nearest-neighbour distance in diamond lattice: a*sqrt(3)/4
    # -----------------------------------------------------------------------
    bond_distance = a * np.sqrt(3) / 4
    created_bonds = set()
 
    crosslink_list = list(crosslinks.values())
    positions      = np.array([cl['pos'] for cl in crosslink_list])
 
    for idx1, cl1 in enumerate(crosslink_list):
        id1   = cl1['id']
        pos1  = cl1['pos']
 
        for idx2, cl2 in enumerate(crosslink_list):
            id2  = cl2['id']
            pos2 = cl2['pos']
 
            if id1 >= id2:
                continue
 
            # Minimum-image distance to handle PBC bonds
            dr = pos2 - pos1
            dr[0] -= box_x * round(dr[0] / box_x)
            dr[1] -= box_y * round(dr[1] / box_y)
            dr[2] -= box_z * round(dr[2] / box_z)
            dist = np.linalg.norm(dr)
 
            if abs(dist - bond_distance) < 0.1 * bond_distance:
                bond_pair = tuple(sorted([id1, id2]))
                if bond_pair in created_bonds:
                    continue
                created_bonds.add(bond_pair)
 
                chain_ids = [id1]
 
                # Interior chain beads interpolated along the minimum-image vector
                for b in range(1, beads_per_chain - 1):
                    frac = b / (beads_per_chain - 1)
                    pos  = pos1 + frac * dr           # stays near pos1 in real space
                    # Wrap into box
                    pos  = pos % np.array([box_x, box_y, box_z])
                    particles.append({
                        'id':   particle_id,
                        'type': 2,            # Chain bead
                        'pos':  pos.copy(),
                        'mol':  molecule_id,
                    })
                    chain_ids.append(particle_id)
                    particle_id += 1
 
                chain_ids.append(id2)
 
                # Assign molecule ID to the two crosslinks of this chain
                particles[id1 - 1]['mol'] = molecule_id
                particles[id2 - 1]['mol'] = molecule_id
 
                for b in range(len(chain_ids) - 1):
                    bonds.append({
                        'id':    bond_id,
                        'type':  1,
                        'atom1': chain_ids[b],
                        'atom2': chain_ids[b + 1],
                    })
                    bond_id += 1
 
                molecule_id += 1
 
    # -----------------------------------------------------------------------
    # Write LAMMPS data file
    # -----------------------------------------------------------------------
    with open(output_file, 'w') as f:
        f.write("LAMMPS data file: pure polymer diamond-lattice gel (no solvent)\n\n")
        f.write(f"{len(particles)} atoms\n")
        f.write(f"{len(bonds)} bonds\n")
        f.write("0 angles\n")
        f.write("0 dihedrals\n")
        f.write("0 impropers\n\n")
        f.write("2 atom types\n")
        f.write("1 bond types\n\n")
        f.write(f"0.0 {box_x:.6f} xlo xhi\n")
        f.write(f"0.0 {box_y:.6f} ylo yhi\n")
        f.write(f"0.0 {box_z:.6f} zlo zhi\n\n")
        f.write("Masses\n\n")
        f.write("1 1.0  # Crosslink\n")
        f.write("2 1.0  # Chain bead\n\n")
        f.write("Atoms\n\n")
        for p in particles:
            f.write(f"{p['id']} {p['mol']} {p['type']} "
                    f"{p['pos'][0]:.6f} {p['pos'][1]:.6f} {p['pos'][2]:.6f}\n")
        f.write("\nBonds\n\n")
        for b in bonds:
            f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")
 
    # -----------------------------------------------------------------------
    # Summary
    # -----------------------------------------------------------------------
    num_crosslinks = sum(1 for p in particles if p['type'] == 1)
    num_chain      = sum(1 for p in particles if p['type'] == 2)
    num_chains     = len(bonds) // (beads_per_chain - 1) if beads_per_chain > 1 else 0
 
    print("Generated pure polymer diamond-lattice gel:")
    print(f"  Unit cells:        {units_x} x {units_y} x {units_z}")
    print(f"  Beads per chain:   {beads_per_chain}  (including crosslink endpoints)")
    print(f"  Lattice constant:  {a:.4f} σ")
    print(f"  Bond distance:     {bond_distance:.4f} σ")
    print(f"  Box dimensions:    {box_x:.4f} x {box_y:.4f} x {box_z:.4f} σ")
    print(f"  Crosslinks:        {num_crosslinks}")
    print(f"  Chain beads:       {num_chain}")
    print(f"  Chains (bonds/4):  {num_chains}")
    print(f"  Total atoms:       {len(particles)}")
    print(f"  Total bonds:       {len(bonds)}")
    print(f"  Output:            {output_file}")


In [2]:
# -----------------------------------------------------------------------
# Inputs
# -----------------------------------------------------------------------
numBeads    = 5
unitsXY     = 8
unitsZ      = 8
outputFile  = "../../lammps_data_files_local/pure_polymer_5beads_8x8x8.data"
 
generate_pure_polymer(numBeads, unitsXY, unitsXY, unitsZ, outputFile)


Generated pure polymer diamond-lattice gel:
  Unit cells:        8 x 8 x 8
  Beads per chain:   5  (including crosslink endpoints)
  Lattice constant:  4.8000 σ
  Bond distance:     2.0785 σ
  Box dimensions:    38.4000 x 38.4000 x 38.4000 σ
  Crosslinks:        4096
  Chain beads:       24576
  Chains (bonds/4):  8192
  Total atoms:       28672
  Total bonds:       32768
  Output:            ../../lammps_data_files_local/pure_polymer_5beads_8x8x8.data
